# Writing & Optimizing GPU/TPU Kernels with JAX Pallas

**What you'll build:** A fused softmax kernel and a tiled matrix multiplication -- real kernels used in production ML systems.

**What you'll learn:**
- The Pallas programming model (grids, tiles, BlockSpecs)
- Why kernel fusion reduces memory bandwidth pressure
- How to benchmark kernels properly on accelerators
- TPU vs GPU optimization strategies

**Prerequisites:** Python, basic linear algebra, curiosity. No CUDA experience needed.

**How to use this notebook:**
1. Start with **TPU** runtime (Runtime > Change runtime type > TPU)
2. Work through Parts 1-9 in order
3. Switch to **GPU** runtime and re-run -- note how the same kernels work but optimal tuning differs

> **Stuck?** There's a [NotebookLM notebook](https://notebooklm.google.com/notebook/c37e48ea-9fba-486c-95ec-dee82f35d3c8) loaded with this codelab, the referenced papers, and the Pallas docs. You can ask it questions as you work through the exercises.

## Part 0: Environment Setup

If you hit version errors when running Pallas kernels, run the install cell below
and restart the runtime. Otherwise, skip it and go straight to imports.


In [ ]:
# Run this cell ONLY if you hit version errors on Pallas kernels.
# Then: Runtime > Restart runtime, skip this cell, continue from imports.
!pip install -U jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html 2>&1 | tail -5


In [ ]:
# Quick version check -- run this after restarting the runtime.
# If the versions are mismatched, re-run the install cell above.
import jax
print(f"JAX version: {jax.__version__}")
try:
    import libtpu
    print(f"libtpu:      {libtpu.__version__}")
except Exception:
    pass  # libtpu version not always inspectable
print(f"Backend:     {jax.default_backend()}")
print(f"Devices:     {jax.devices()}")

_backend = jax.default_backend()
if _backend == "tpu":
    print("TPU ready.")
elif _backend == "gpu":
    print("GPU ready. Some TPU-specific cells will be skipped automatically.")
else:
    print(f"WARNING: Running on {_backend.upper()}. Kernels will use interpret=True")
    print("(Python fallback). Switch to TPU or GPU for real performance.")

In [ ]:
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
from jax import random
import functools
import time
import numpy as np

backend = jax.default_backend()
IS_TPU = backend == "tpu"
IS_GPU = backend == "gpu"
IS_CPU = backend == "cpu"

# Pallas GPU backend requires Ampere (compute capability 8.0+).
# Older GPUs (T4, V100) must use interpret mode.
USE_INTERPRET = IS_CPU
if IS_GPU:
    try:
        cc = jax.devices()[0].compute_capability
        if int(cc.split(".")[0]) < 8:
            print(f"GPU compute capability {cc} < 8.0 (need Ampere+). Using interpret mode.")
            USE_INTERPRET = True
    except Exception:
        pass

# Import TPU-specific Pallas module (used in Parts 6-8)
if IS_TPU:
    from jax.experimental.pallas import tpu as pltpu

print(f"JAX {jax.__version__} on {backend.upper()}")
if IS_CPU:
    print("WARNING: You're on CPU. Kernels work with interpret=True but")
    print("won't benchmark realistically. Switch to GPU or TPU runtime.")

## Part 1: The Mental Model

Think of an accelerator as a system that processes data in **tiles**:
1. **Grab a tile** of the input data from main memory (HBM) into fast local memory (SRAM / shared mem)
2. **Do math** on that tile
3. **Write the result** back to main memory

**Pallas lets you define three things:**

| Concept | What it does | Systems analogy |
|---|---|---|
| **Kernel** | The math to perform on each tile | The per-shard query |
| **Grid** | How many tiles to process | Shard count |
| **BlockSpec** | Which tile each grid index maps to | Shard key / partition function |

> **GPU vs TPU execution:** On GPUs, grid programs run **concurrently** on different SMs (streaming multiprocessors) -- the "warehouse of workers" model. On TPUs, the compiler **pipelines** grid iterations on a single core, overlapping data fetching with compute. Same Pallas code, very different hardware strategies. You don't need to think about this yet, but it explains why tuning advice differs between backends.

> **Tip:** You can run any Pallas kernel on CPU for debugging by passing `interpret=True` to `pallas_call`. This interprets the kernel in Python instead of compiling to accelerator code.

## Part 2: Hello World -- Vector Addition

### Version 1: No tiling
The entire vector is processed in one shot. Shows the API without any complexity.

In [ ]:
def add_kernel(x_ref, y_ref, o_ref):
    """The kernel function.

    x_ref, y_ref are inputs; o_ref is the output.
    These are Ref objects -- think of them as pointers to tiles in fast local memory.
    x_ref[...] reads the whole tile. You can also slice: x_ref[0:4].
    """
    o_ref[...] = x_ref[...] + y_ref[...]


def vector_add_basic(x, y):
    return pl.pallas_call(
        add_kernel,                                          # the kernel
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),   # output metadata
        interpret=USE_INTERPRET,                                    # CPU fallback
    )(x, y)


x = jnp.arange(8, dtype=jnp.float32)
y = jnp.ones(8, dtype=jnp.float32)
result = vector_add_basic(x, y)

print(f"x:       {x}")
print(f"y:       {y}")
print(f"x + y:   {result}")
print(f"Correct: {jnp.allclose(result, x + y)}")

### Version 2: Tiled

Now we split the vector into blocks. This is how real kernels work -- you can't fit a 10 GB tensor into SRAM.

### How BlockSpec works

**1D example:**
```
Vector x: [x0 x1 x2 x3 | x4 x5 x6 x7 | x8 x9 x10 x11 | x12 x13 x14 x15]
           ────────────   ─────────────   ──────────────   ────────────────
           Block 0 (i=0)  Block 1 (i=1)   Block 2 (i=2)   Block 3 (i=3)

BlockSpec(block_shape=(4,), index_map=lambda i: (i,))

  i=0 → start at element 0*4=0,  take 4 elements → [x0,  x1,  x2,  x3]
  i=1 → start at element 1*4=4,  take 4 elements → [x4,  x5,  x6,  x7]
  i=2 → start at element 2*4=8,  take 4 elements → [x8,  x9,  x10, x11]
  i=3 → start at element 3*4=12, take 4 elements → [x12, x13, x14, x15]

The index_map returns one index per dimension. Pallas multiplies each
by the corresponding block_shape element to get the element offset.
So (i,) with block_shape (4,) → start at element i * 4.
```

**2D example** (used in softmax, matmul, and beyond):
```
Matrix x (4 rows × 8 cols), block_shape=(2, 8):

  Row 0  [ . . . . . . . . ]  ← Block 0 (i=0): rows 0-1, all cols
  Row 1  [ . . . . . . . . ]
  Row 2  [ . . . . . . . . ]  ← Block 1 (i=1): rows 2-3, all cols
  Row 3  [ . . . . . . . . ]

BlockSpec(block_shape=(2, 8), index_map=lambda i: (i, 0))

  i=0 → start at (0*2, 0*8) = (0, 0) → rows 0-1, cols 0-7
  i=1 → start at (1*2, 0*8) = (2, 0) → rows 2-3, cols 0-7

The lambda returns (i, 0): i tiles over rows, 0 means "always
start at column 0" -- we take the full row width each time.
```

This 2D pattern -- tile one dimension, take the full extent of another -- is the workhorse of softmax, LayerNorm, and row-wise kernels.

In [ ]:
BLOCK_SIZE = 1024
def vector_add_tiled(x, y, block_size=BLOCK_SIZE):
    n = x.shape[0]
    assert n % block_size == 0, f"Size {n} must be divisible by block_size {block_size}"

    return pl.pallas_call(
        add_kernel,  # Same kernel as before! Tiling is orthogonal to the math.
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n // block_size,),                    # number of blocks to launch
        in_specs=[
            # BlockSpec(block_shape, index_map)
            # index_map: given grid index i, return the starting indices of this block
            pl.BlockSpec((block_size,), lambda i: (i,)),
            pl.BlockSpec((block_size,), lambda i: (i,)),
        ],
        out_specs=pl.BlockSpec((block_size,), lambda i: (i,)),
        interpret=USE_INTERPRET,
    )(x, y)


n = 8192
x = jnp.arange(n, dtype=jnp.float32)
y = jnp.ones(n, dtype=jnp.float32)
result = vector_add_tiled(x, y, block_size=BLOCK_SIZE)

grid_size = n // BLOCK_SIZE
print(f"Vector size: {n}")
print(f"Block size:  {BLOCK_SIZE}")
print(f"Grid size:   {grid_size}  ({grid_size} tile operations)")
print(f"Correct:     {jnp.allclose(result, x + y)}")

## Part 3: Exercise -- Write an Element-wise Kernel

Write a Pallas kernel for **ReLU**: `f(x) = max(x, 0)`.

This time, work with a **2D input** (a matrix), tiling over rows. This is the same pattern used by softmax and LayerNorm -- tile one dimension, take the full extent of the other. Refer to the 2D BlockSpec example above.

In [ ]:
# ============================================================
# YOUR CODE HERE
# ============================================================

def relu_kernel(x_ref, o_ref):
    o_ref[...] = jnp.maximum(x_ref[...], 0)

def pallas_relu(x, block_rows=8):
    n_rows, n_cols = x.shape
    assert n_rows % block_rows == 0
    return pl.pallas_call(
        relu_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=USE_INTERPRET,
    )(x)


# ---------- test harness (don't modify) ----------
key = random.PRNGKey(0)
x = random.normal(key, (256, 512))
try:
    result = pallas_relu(x)
    expected = jnp.maximum(x, 0)
    if jnp.allclose(result, expected):
        print("Correct! Your 2D ReLU kernel works.")
    else:
        print("Output doesn't match. Check your kernel.")
        print(f"  Expected (row 0, first 8): {expected[0, :8]}")
        print(f"  Got      (row 0, first 8): {result[0, :8]}")
except NotImplementedError:
    print("Implement relu_kernel and pallas_relu above, then re-run.")

#### Solution

```python
def relu_kernel(x_ref, o_ref):
    o_ref[...] = jnp.maximum(x_ref[...], 0)

def pallas_relu(x, block_rows=8):
    n_rows, n_cols = x.shape
    return pl.pallas_call(
        relu_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=USE_INTERPRET,
    )(x)
```

## Part 4: Exercise -- Fused Softmax

Softmax is in every transformer attention layer. Let's understand why a custom kernel matters, then write one.

### Why fusion matters

Standard softmax compiles to **3 separate passes** over your data in HBM:

| Pass | Read from HBM | Compute | Write to HBM |
|------|--------------|---------|-------------|
| 1 | all elements | `max(x)` | the max |
| 2 | all elements + max | `exp(x - max)`, `sum(...)` | exp values, the sum |
| 3 | all elements + sum | `exp(x - max) / sum` | final result |

That's **3 full reads** of the tensor from HBM. Each HBM read is slow and expensive.

A **fused kernel** does it in **1 read + 1 write**:
1. Load a block of rows into SRAM
2. Compute max, exp, sum, divide -- all in SRAM (fast!)
3. Write the result back to HBM

For large matrices, softmax is **memory-bandwidth bound** -- the arithmetic is cheap; moving data is the bottleneck. Cutting memory traffic by ~3x gives a real speedup.

> **Systems analogy:** It's the same insight as batching database writes -- one round-trip is better than three, even if each round-trip carries more data.

Now write it yourself. The kernel needs to be **numerically stable** -- subtract the row max before exponentiating to avoid overflow.

In [ ]:
# Baseline: standard JAX softmax (for comparison)
@jax.jit
def jax_softmax(x):
    return jax.nn.softmax(x, axis=-1)

def softmax_kernel(x_ref, o_ref):
    row = x_ref[...]
    row_max = jnp.max(row, axis=-1, keepdims=True)
    safe_row = row - row_max
    numerator = jnp.exp(safe_row)
    denominator = jnp.sum(numerator, axis=-1, keepdims=True)
    o_ref[...] = numerator / denominator

def fused_softmax(x, block_rows=8):
    n_rows, n_cols = x.shape
    assert n_rows % block_rows == 0
    return pl.pallas_call(
        softmax_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=USE_INTERPRET,
    )(x)


# ---------- test harness (don't modify) ----------
key = random.PRNGKey(42)
x = random.normal(key, (1024, 512), dtype=jnp.float32)

try:
    result_pallas = fused_softmax(x)
    result_jax = jax_softmax(x)
    max_err = jnp.max(jnp.abs(result_jax - result_pallas))
    if jnp.any(jnp.isnan(result_pallas)):
        print('Output contains NaN! Did you forget to subtract the max before exp?')
    elif jnp.allclose(result_jax, result_pallas, atol=1e-5):
        print(f"Correct! Max error: {max_err:.2e}")
        print(f"Row sums [0:5]: {jnp.sum(result_pallas, axis=-1)[:5]}  (should be ~1.0)")
    else:
        print(f"Wrong answer. Max error: {max_err:.2e}")
        print(f"Row sums [0:5]: {jnp.sum(result_pallas, axis=-1)[:5]}  (should be ~1.0)")
except NotImplementedError:
    print("Implement softmax_kernel and fused_softmax above, then re-run.")

#### Solution

```python
def softmax_kernel(x_ref, o_ref):
    row = x_ref[...]
    row_max = jnp.max(row, axis=-1, keepdims=True)
    safe_row = row - row_max
    numerator = jnp.exp(safe_row)
    denominator = jnp.sum(numerator, axis=-1, keepdims=True)
    o_ref[...] = numerator / denominator

def fused_softmax(x, block_rows=8):
    n_rows, n_cols = x.shape
    assert n_rows % block_rows == 0
    return pl.pallas_call(
        softmax_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=USE_INTERPRET,
    )(x)
```

**Important:** The rest of the notebook uses `fused_softmax` in benchmarks and later exercises. After completing this exercise (or copying the solution), make sure it runs successfully before continuing.

## Part 5: Benchmarking

**Rule #1** of benchmarking on accelerators: always call `.block_until_ready()`. JAX (via its compiler, XLA) dispatches asynchronously -- without this, you're timing the dispatch overhead, not the actual compute.

In [ ]:
def benchmark(fn, *args, warmup=10, trials=50, label=""):
    """Benchmark with proper warmup and sync."""
    # Warmup: JIT compilation + cache settling
    for _ in range(warmup):
        result = fn(*args)
        result.block_until_ready()

    times_ms = []
    for _ in range(trials):
        start = time.perf_counter()
        result = fn(*args)
        result.block_until_ready()
        times_ms.append((time.perf_counter() - start) * 1000)

    t = np.array(times_ms)
    if label:
        print(f"  {label:35s} "
              f"median={np.median(t):8.3f}ms  "
              f"mean={np.mean(t):8.3f}ms  "
              f"min={np.min(t):8.3f}ms  "
              f"std={np.std(t):7.3f}ms")
    return {'median': np.median(t), 'mean': np.mean(t), 'min': np.min(t)}

In [ ]:
print(f"Softmax benchmark on {backend.upper()}")
print("=" * 90)

for n_rows, n_cols in [(1024, 512), (4096, 1024), (8192, 2048)]:
    print(f"\nMatrix: {n_rows} x {n_cols}")
    print("-" * 90)
    x = random.normal(random.PRNGKey(0), (n_rows, n_cols), dtype=jnp.float32)

    benchmark(jax_softmax, x, label="jax.nn.softmax (XLA)")
    benchmark(fused_softmax, x, label="Pallas fused softmax")

**What to look for:**
- At small sizes, XLA may match or beat us (kernel launch overhead dominates)
- At large sizes, the fused kernel should win (memory bandwidth dominates)
- If Pallas is slower, don't worry -- XLA is very good! The point is understanding *how* kernels work, and for custom ops XLA can't fuse, this is how you beat it

## Part 6: Inside the Machine -- Hardware-Aware Optimization

Tweaking `block_rows` is the **last** step of kernel optimization. The real work is designing your algorithm around the hardware's physical architecture.

This section covers:
1. **Roofline analysis** -- is my kernel memory-bound or compute-bound?
2. **MXU exploitation** -- feeding the matrix unit what it wants
3. **Pipelining** -- overlapping data movement with computation
4. **Fused LayerNorm** -- a second real kernel that applies all of these ideas


### The Roofline Model: Memory-Bound vs Compute-Bound

Before optimizing, you need to know **what the bottleneck is.** The roofline model gives you a quick answer.

**Arithmetic Intensity (AI)** = FLOPs performed / Bytes moved from HBM

Every chip has a **ridge point** = Peak FLOPS / Peak HBM Bandwidth. This is the crossover:
- AI < ridge point → **memory-bound** → optimization goal: *reduce data movement* (fusion, data reuse)
- AI > ridge point → **compute-bound** → optimization goal: *maximize hardware utilization* (MXU-friendly tiles, bf16)

> **Systems analogy:** It's the same as diagnosing whether your service is CPU-bound or I/O-bound. If you're I/O-bound, optimizing your algorithm's Big-O doesn't help -- you need to reduce round-trips (batching, caching). If you're CPU-bound, caching doesn't help -- you need a faster algorithm or better hardware utilization.


In [ ]:
# Roofline analysis for our two kernels
# (Using approximate TPU v5e specs -- exact values vary by chip revision)
# NOTE: TPU v6e (Trillium) is now available with ~2x HBM bandwidth and
# ~4.7x peak compute over v5e. The analysis approach is the same --
# just plug in the new specs.

HBM_BW = 820       # GB/s (v5e)
PEAK_FLOPS = 197    # TFLOPS bf16 matmul on MXU (v5e)
RIDGE_POINT = (PEAK_FLOPS * 1e12) / (HBM_BW * 1e9)

# NOTE: PEAK_FLOPS above is for the MXU (matrix unit), which handles matmuls.
# Element-wise ops (softmax, LayerNorm) run on the vector unit, which has much
# lower peak throughput (~10-20 TFLOPS depending on dtype and chip revision).
# We use MXU peak here because it determines the ridge point for matmul.
# Softmax is so deeply memory-bound that the exact vector-unit peak doesn't
# change the diagnosis.

print("TPU v5e (approximate)")
print(f"  HBM bandwidth:       {HBM_BW} GB/s")
print(f"  MXU peak (bf16):     {PEAK_FLOPS} TFLOPS")
print(f"  Ridge point:         {RIDGE_POINT:.0f} FLOPS/byte")
print()

# --- Softmax ---
# Per row of length D: ~5D effective FLOPs (max, sub, exp, sum, div)
# (exp is really a multi-instruction sequence, but the exact count doesn't
#  matter -- softmax is so deeply memory-bound that even 10x more FLOPs
#  wouldn't change the diagnosis.)
# Data moved: read D floats + write D floats = 2D * 4 bytes (fp32)
D = 2048
sm_ai = (5 * D) / (2 * D * 4)
print(f"Softmax (row length {D}):")
print(f"  Arithmetic intensity: {sm_ai:.2f} FLOPS/byte  (approximate)")
print(f"  Ridge point:          {RIDGE_POINT:.0f} FLOPS/byte")
print(f"  Diagnosis:            MEMORY-BOUND  ({sm_ai:.2f} << {RIDGE_POINT:.0f})")
print(f"  Strategy:             Reduce data movement -> FUSION")
print()

# --- Matrix multiply ---
# C[M,N] = A[M,K] @ B[K,N]: 2*M*N*K FLOPs
# Data moved (ideal, assuming perfect tiling with maximum reuse):
#   read A + read B + write C = (MK + KN + MN) * bytes_per_elem
# In practice, each A-tile is read N/bn times and each B-tile M/bm times.
# Larger tiles -> more data reuse -> closer to this ideal.
M = N = K = 2048
mm_flops = 2 * M * N * K
mm_bytes = (M*K + K*N + M*N) * 2    # bf16 = 2 bytes
mm_ai = mm_flops / mm_bytes
print(f"Matmul ({M}x{K} @ {K}x{N}, bf16):")
print(f"  Arithmetic intensity: {mm_ai:.0f} FLOPS/byte  (ideal, perfect tiling)")
print(f"  Ridge point:          {RIDGE_POINT:.0f} FLOPS/byte")
print(f"  Diagnosis:            COMPUTE-BOUND  ({mm_ai:.0f} > {RIDGE_POINT:.0f})")
print(f"  Strategy:             Maximize MXU utilization -> bf16, 128-aligned tiles")

### Exploiting the Matrix Unit (MXU)

The TPU's MXU is a **128×128 systolic array** -- a grid of multiply-accumulate units that streams data through in a pipeline. A full 128×128 bf16 matmul takes ~128 cycles to fill and drain the array.

**What happens when you don't match the MXU:**

| Mismatch | Consequence |
|----------|------------|
| **fp32 instead of bf16** | Input data is 2× larger, reducing feed throughput. Actual slowdown depends on chip generation (2-4×). Always prefer bf16 for matmuls when precision allows. |
| **Tile not a multiple of 128** | Compiler pads the tile with zeros. You pay for a full 128×128 but only use part of it. |
| **Tile too small (e.g., 32×32)** | Uses only ~6% of the MXU array. The real cliff is going below 128 on the MXU-contracted dimension. |

> **Production tip:** Real kernels almost always accept **bf16 inputs**, accumulate internally in **fp32** for precision, and cast the output back to **bf16**. This notebook uses fp32 throughout for simplicity, but mixed-precision is the standard pattern.

Let's measure this directly. We'll provide a working K-tiled matmul for benchmarking -- you'll implement your own in Part 7.

In [ ]:
# Utility matmul for benchmarking -- uses concepts from Parts 7-8.
# Don't try to understand this code yet. You'll learn each piece when
# you implement your own matmul. (you'll write your own in Part 7)
def _matmul_kernel(x_ref, y_ref, o_ref):
    # NOTE: += works because Pallas zero-initializes the output buffer
    # on TPU and GPU. This is the standard accumulation pattern for
    # reduction kernels (tiling over the K dimension).
    # On TPU, matmul accumulation MUST be in fp32 (MXU requirement).
    @pl.when(pl.program_id(2) == 0)
    def _init():
        o_ref[...] = jnp.zeros_like(o_ref[...])
    x = x_ref[...].astype(jnp.float32)
    y = y_ref[...].astype(jnp.float32)
    o_ref[...] += x @ y

def _bench_matmul(x, y, bm=128, bn=128, bk=128):
    m, k = x.shape
    _, n = y.shape
    # Output must be fp32 for TPU MXU accumulation, even with bf16 inputs
    out_dtype = jnp.float32
    compiler_params = {}
    if IS_TPU:
        compiler_params = pltpu.CompilerParams(
            dimension_semantics=["parallel", "parallel", "arbitrary"]
        )
    return pl.pallas_call(
        _matmul_kernel,
        out_shape=jax.ShapeDtypeStruct((m, n), out_dtype),
        grid=(m // bm, n // bn, k // bk),
        in_specs=[
            pl.BlockSpec((bm, bk), lambda i, j, k: (i, k)),
            pl.BlockSpec((bk, bn), lambda i, j, k: (k, j)),
        ],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j, k: (i, j)),
        compiler_params=compiler_params,
        interpret=USE_INTERPRET,
    )(x, y)

# Verify
a = random.normal(random.PRNGKey(0), (512, 512), dtype=jnp.bfloat16)
b = random.normal(random.PRNGKey(1), (512, 512), dtype=jnp.bfloat16)
result = _bench_matmul(a, b)
ref = a.astype(jnp.float32) @ b.astype(jnp.float32)
err = jnp.max(jnp.abs(result - ref))
print(f"Utility matmul check: max error {err:.2e}")


In [ ]:
# === MXU Benchmark: dtype and tile-size impact ===
print("MXU Exploitation Benchmark")
print("=" * 90)
size = 2048

# --- Experiment 1: bf16 vs fp32 ---
print(f"\nExperiment 1: Data type (matmul {size}x{size})")
print("-" * 90)

a_bf16 = random.normal(random.PRNGKey(0), (size, size), dtype=jnp.bfloat16)
b_bf16 = random.normal(random.PRNGKey(1), (size, size), dtype=jnp.bfloat16)
a_fp32 = a_bf16.astype(jnp.float32)
b_fp32 = b_bf16.astype(jnp.float32)

jit_mm = jax.jit(jnp.matmul)
benchmark(jit_mm, a_bf16, b_bf16, label="jnp.matmul bf16 (baseline)")
benchmark(jit_mm, a_fp32, b_fp32, label="jnp.matmul fp32 (baseline)")
benchmark(lambda a, b: _bench_matmul(a, b, 128, 128, 128), a_bf16, b_bf16,
          label="Pallas bf16 128x128 (MXU sweet spot)")
benchmark(lambda a, b: _bench_matmul(a, b, 128, 128, 128), a_fp32, b_fp32,
          label="Pallas fp32 128x128")

# --- Experiment 2: tile sizes (bf16) ---
print(f"\nExperiment 2: Tile size with bf16 ({size}x{size})")
print("-" * 90)
for tile in [128, 256, 512]:
    if size % tile != 0:
        continue
    try:
        benchmark(lambda a, b, t=tile: _bench_matmul(a, b, t, t, t),
                  a_bf16, b_bf16, label=f"tile={tile}x{tile}")
    except Exception as e:
        print(f"  tile={tile}x{tile}: FAILED -- {type(e).__name__}: {e}")


### Pipelining: Hiding Memory Latency

When the MXU is multiplying the current tile, you want the DMA engine to **already be fetching the next tile** from HBM into VMEM. This overlap is called **pipelining**.

```
Without pipelining:        With pipelining:
┌──────┐                   ┌──────┐
│ DMA  │ fetch tile 0      │ DMA  │ fetch tile 0
├──────┤                   ├──────┼──────┐
│ MXU  │ compute tile 0    │ DMA  │ MXU  │ fetch tile 1 + compute tile 0
├──────┤                   ├──────┼──────┤
│ DMA  │ fetch tile 1      │ DMA  │ MXU  │ fetch tile 2 + compute tile 1
├──────┤                   ├──────┼──────┤
│ MXU  │ compute tile 1    │      │ MXU  │ compute tile 2
└──────┘                   └──────┴──────┘
Total: 4 steps             Total: ~2.5 steps (DMA hidden behind compute)
```

In Pallas, you enable pipelining by telling the compiler which grid dimensions are **reductions** vs **independent**:

```python
from jax.experimental.pallas import tpu as pltpu

compiler_params = pltpu.CompilerParams(
    dimension_semantics=["parallel", "parallel", "arbitrary"]
)
```

- `"parallel"` means independent output tiles (i, j). Can run on different cores (megacore).
- `"arbitrary"` means the compiler may reorder and pipeline iterations along this dimension. Used for reductions, but more generally means "iterations along this dimension may read/write overlapping output regions." The dimension must not have loop-carried dependencies beyond the accumulator pattern (`o_ref[...] += ...`).

**More K-tiles = more pipeline stages = better latency hiding** (up to a point, there's startup/drain overhead).

> **`fori_loop` vs grid-based iteration:** You can also loop inside the kernel with `jax.lax.fori_loop` (we'll see this in Flash Attention). The tradeoff: `fori_loop` gives explicit control but **prevents** the compiler from pipelining DMA. Grid-based iteration enables pipelining but requires scratch space for cross-iteration state. Prefer the grid when DMA/compute overlap matters (matmul, attention).

In [ ]:
# === Pipelining Benchmark: K-tile size sweep ===
# More K-tiles → more pipeline stages → better DMA/compute overlap
print("Pipelining Benchmark: K-tile size sweep")
print("=" * 90)

size = 2048
a = random.normal(random.PRNGKey(0), (size, size), dtype=jnp.bfloat16)
b = random.normal(random.PRNGKey(1), (size, size), dtype=jnp.bfloat16)

print(f"Matmul {size}x{size} bf16, output tiles fixed at 128x128")
print(f"Varying bk (K-tile size) → changes number of pipeline stages")
print("-" * 90)

results = []
for bk in [2048, 1024, 512, 256, 128]:
    if size % bk != 0:
        continue
    n_stages = size // bk
    try:
        stats = benchmark(
            lambda a, b, _bk=bk: _bench_matmul(a, b, 128, 128, _bk),
            a, b, warmup=5, trials=20,
            label=f"bk={bk:4d} ({n_stages:3d} K-iterations)"
        )
        results.append((bk, size // bk, stats['median']))
    except Exception as e:
        print(f"  bk={bk}: FAILED -- {type(e).__name__}: {e}")

if len(results) > 1:
    best = min(results, key=lambda r: r[2])
    worst = max(results, key=lambda r: r[2])
    print(f"\nBest:  bk={best[0]} ({best[1]} stages, {best[2]:.3f}ms)")
    print(f"Worst: bk={worst[0]} ({worst[1]} stages, {worst[2]:.3f}ms)")
    if worst[2] > 0:
        print(f"Speedup from pipelining: {worst[2]/best[2]:.2f}x")


### Putting It All Together: Fused LayerNorm

Let's apply everything to a second real kernel. **LayerNorm** is used in every transformer layer and, like softmax, is memory-bound:

```
LayerNorm(x) = (x - mean(x)) / sqrt(var(x) + eps) * weight + bias
```

**Without fusion (4 passes over HBM):**
1. Read x → compute mean → write mean
2. Read x + mean → compute variance → write variance
3. Read x + mean + var → normalize → write normalized
4. Read normalized + weight + bias → scale and shift → write output

**With fusion (1 read + 1 write):**
1. Load a block of rows into SRAM
2. Compute mean, variance, normalize, scale, shift -- all in SRAM
3. Write the result

Same pattern as softmax, but with more ops fused together. This is the bread and butter of kernel optimization: **load once, compute everything, store once.**


In [ ]:
# Fused LayerNorm kernel

def layernorm_kernel(x_ref, weight_ref, bias_ref, o_ref):
    """Fused LayerNorm: mean + variance + normalize + affine, one SRAM round-trip."""
    x = x_ref[...]           # (block_rows, D) -- loaded into SRAM once
    w = weight_ref[...]      # (D,) -- broadcast across rows
    b = bias_ref[...]        # (D,)

    # All of this math happens in SRAM -- no HBM traffic until the final store
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.mean(jnp.square(x - mean), axis=-1, keepdims=True)
    x_norm = (x - mean) / jnp.sqrt(var + 1e-5)
    o_ref[...] = x_norm * w + b


def fused_layernorm(x, weight, bias, block_rows=8):
    n_rows, n_cols = x.shape
    assert n_rows % block_rows == 0

    return pl.pallas_call(
        layernorm_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[
            pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),  # x: tiled
            pl.BlockSpec((n_cols,), lambda i: (0,)),               # weight: same for all tiles
            pl.BlockSpec((n_cols,), lambda i: (0,)),               # bias: same for all tiles
        ],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=USE_INTERPRET,
    )(x, weight, bias)


# Verify correctness against JAX's implementation
def jax_layernorm(x, weight, bias, eps=1e-5):
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.mean(jnp.square(x - mean), axis=-1, keepdims=True)
    return (x - mean) / jnp.sqrt(var + eps) * weight + bias

key = random.PRNGKey(42)
x = random.normal(key, (1024, 512), dtype=jnp.float32)
w = random.normal(random.PRNGKey(1), (512,), dtype=jnp.float32)
b = random.normal(random.PRNGKey(2), (512,), dtype=jnp.float32)

result_jax = jax_layernorm(x, w, b)
result_pallas = fused_layernorm(x, w, b)

print(f"Input shape:    {x.shape}")
print(f"Max abs error:  {jnp.max(jnp.abs(result_jax - result_pallas)):.2e}")
print(f"Correct:        {jnp.allclose(result_jax, result_pallas, atol=1e-4)}")


In [ ]:
# Benchmark fused LayerNorm vs JAX
print(f"LayerNorm benchmark on {backend.upper()}")
print("=" * 90)

jit_ln = jax.jit(jax_layernorm)

for n_rows, n_cols in [(1024, 512), (4096, 1024), (8192, 2048)]:
    print(f"\nMatrix: {n_rows} x {n_cols}")
    print("-" * 90)

    x = random.normal(random.PRNGKey(0), (n_rows, n_cols), dtype=jnp.float32)
    w = jnp.ones(n_cols, dtype=jnp.float32)
    b = jnp.zeros(n_cols, dtype=jnp.float32)

    benchmark(jit_ln, x, w, b, label="JAX LayerNorm (XLA)")

    # Try a few block sizes
    for br in [8, 32, 128]:
        if n_rows % br == 0:
            try:
                benchmark(
                    lambda x, w, b, _br=br: fused_layernorm(x, w, b, block_rows=_br),
                    x, w, b, label=f"Pallas fused (block_rows={br})"
                )
            except Exception as e:
                print(f"  block_rows={br}: FAILED -- {type(e).__name__}")


### The Last Step: Parameter Tuning

Now that you understand *what* to optimize and *why*, the final step is sweeping parameters to find the sweet spot for your specific hardware and input shapes.

For **softmax** the main knob is `block_rows`. For **matmul** it's `(bm, bn, bk)`. The optimal values depend on:
- VMEM capacity (bigger tiles = more data reuse, but must fit in SRAM)
- MXU dimensions (tiles should be multiples of 128 for bf16)
- Available parallelism (more tiles in the grid = better utilization of multiple cores)


In [ ]:
# Softmax block-size sweep (same code as before, now in context)
print(f"Softmax block-size sweep on {backend.upper()}")
print(f"Matrix: 8192 x 2048\n")

x = random.normal(random.PRNGKey(0), (8192, 2048), dtype=jnp.float32)

block_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]

results = []
for bs in block_sizes:
    if 8192 % bs != 0:
        continue
    try:
        stats = benchmark(
            lambda x, _bs=bs: fused_softmax(x, block_rows=_bs), x,
            warmup=5, trials=20, label=f"block_rows={bs}"
        )
        results.append((bs, stats['median']))
    except Exception as e:
        print(f"  block_rows={bs:4d}: FAILED -- {type(e).__name__}: {e}")

if results:
    best_bs, best_time = min(results, key=lambda r: r[1])
    print(f"\nBest: block_rows={best_bs} ({best_time:.3f}ms)")


### Quick Reference: TPU vs GPU Tuning

| | TPU | GPU |
|---|---|---|
| **Sweet-spot tile size** | Multiples of 128 (MXU) | Multiples of 32 (warp size) |
| **Preferred dtype** | bfloat16 | float16 or bfloat16 |
| **Pipelining control** | `pltpu.CompilerParams(dimension_semantics=...)` | Backend-specific (Mosaic GPU is default, Triton available) |
| **Parallelism control** | Megacore (multi-core) | `num_warps` (Triton backend) |
| **SRAM size** | ~16-32 MB VMEM per chip | ~100-200 KB shared mem per SM |
| **Key bottleneck** | HBM bandwidth (memory-bound ops) | HBM bandwidth + occupancy |

> **Note (March 2026):** The default Pallas GPU backend is now **Mosaic GPU**, not Triton. Triton is still available via `jax.experimental.pallas.triton`. The tuning knobs differ between backends.

## Part 7: Boss Fight -- Matrix Multiplication

Matmul is the most important kernel in ML. Even if you never ship a custom one, understanding tiled matmul teaches you:
- **2D tiling** (block rows AND columns)
- **Reduction dimensions** (accumulating partial products across K)
- **Data reuse** (each A-tile is used for every B-tile in the same row, and vice versa)

```
C[M, N] = A[M, K] @ B[K, N]

Tiled:
  for each output tile C[i, j]:       # grid dimensions i, j
      for k in range(K // bk):        # reduction dimension
          C[i, j] += A[i, k] @ B[k, j]
```

### Level 1: Basic matmul (no K-tiling)

Each kernel invocation loads a full row-strip of A and a full column-strip of B. Simple but loads too much data per tile.

In [ ]:
# ============================================================
# YOUR CODE HERE: Implement tiled matrix multiplication
# ============================================================

def matmul_kernel(x_ref, y_ref, o_ref):
    o_ref[...] = x_ref[...] @ y_ref[...]

def pallas_matmul(x, y, bm=128, bn=128):
    m, k = x.shape
    _, n = y.shape
    return pl.pallas_call(
        matmul_kernel,
        out_shape=jax.ShapeDtypeStruct((m, n), x.dtype),
        grid=(m // bm, n // bn),
        in_specs=[
            pl.BlockSpec((bm, k), lambda i, j: (i, 0)),
            pl.BlockSpec((k, bn), lambda i, j: (0, j)),
        ],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j: (i, j)),
        interpret=USE_INTERPRET,
    )(x, y)


# ---------- test harness ----------
a = random.normal(random.PRNGKey(0), (512, 256), dtype=jnp.float32)
b = random.normal(random.PRNGKey(1), (256, 512), dtype=jnp.float32)

try:
    result = pallas_matmul(a, b)
    expected = a @ b
    max_err = jnp.max(jnp.abs(result - expected))
    if jnp.allclose(result, expected, atol=1e-3):
        print(f"Matmul correct! Max error: {max_err:.2e}")
    else:
        print(f"Wrong answer. Max error: {max_err:.2e}")
except NotImplementedError:
    print("Implement matmul_kernel and pallas_matmul above.")

#### Solution

```python
def matmul_kernel(x_ref, y_ref, o_ref):
    o_ref[...] = x_ref[...] @ y_ref[...]

def pallas_matmul(x, y, bm=128, bn=128):
    m, k = x.shape
    _, n = y.shape
    return pl.pallas_call(
        matmul_kernel,
        out_shape=jax.ShapeDtypeStruct((m, n), x.dtype),
        grid=(m // bm, n // bn),
        in_specs=[
            pl.BlockSpec((bm, k), lambda i, j: (i, 0)),   # row strip of A
            pl.BlockSpec((k, bn), lambda i, j: (0, j)),   # col strip of B
        ],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j: (i, j)),
        interpret=USE_INTERPRET,
    )(x, y)
```

### Level 2: K-Tiled Matmul

The Level 1 matmul loads the entire K dimension per tile. If K=4096, that's a lot of data in SRAM per invocation.

**K-tiling** splits the reduction dimension into blocks and accumulates partial results:

```
C[i, j] = 0
for k in range(K // bk):
    C[i, j] += A[i, k] @ B[k, j]   # each A[i,k] is (bm, bk), B[k,j] is (bk, bn)
```

On **TPU**, we add K to the grid and use `dimension_semantics` to mark it as a reduction ("arbitrary") dimension. The TPU compiler then:
- Pipelines the K iterations (prefetching the next A/B tiles while computing the current ones)
- Manages the accumulator automatically

**Important:** The `+=` accumulation pattern (`o_ref[...] += x_ref[...] @ y_ref[...]`) works because the output buffer starts at zero (we ensure this with a `pl.when` guard on the first iteration). Multiple grid iterations with `"arbitrary"` semantics write to the same output tile, and the `+=` accumulates partial products. Using `=` instead of `+=` would overwrite previous iterations -- a common bug.

In [ ]:
def matmul_kernel_acc(x_ref, y_ref, o_ref):
    """Accumulate one (bm, bk) @ (bk, bn) partial product."""
    @pl.when(pl.program_id(2) == 0)
    def _init():
        o_ref[...] = jnp.zeros_like(o_ref[...])
    x = x_ref[...].astype(jnp.float32)
    y = y_ref[...].astype(jnp.float32)
    o_ref[...] += x @ y


def pallas_matmul_k_tiled(x, y, bm=128, bn=128, bk=128):
    """K-tiled matrix multiply.

    Grid: (M // bm, N // bn, K // bk)
      - i, j are "parallel": each (i, j) pair produces an independent output tile
      - k is "arbitrary" (reduction): all k values contribute to the same output tile
    """
    m, k = x.shape
    _, n = y.shape
    assert m % bm == 0 and n % bn == 0 and k % bk == 0

    # On TPU, dimension_semantics tells the compiler which dims are reductions.
    # This enables pipelining: fetch next K-tile while computing current one.
    compiler_params = {}
    if IS_TPU:
        compiler_params = pltpu.CompilerParams(
            dimension_semantics=["parallel", "parallel", "arbitrary"]
        )

    return pl.pallas_call(
        matmul_kernel_acc,
        out_shape=jax.ShapeDtypeStruct((m, n), x.dtype),
        grid=(m // bm, n // bn, k // bk),
        in_specs=[
            pl.BlockSpec((bm, bk), lambda i, j, k: (i, k)),  # A tile
            pl.BlockSpec((bk, bn), lambda i, j, k: (k, j)),  # B tile
        ],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j, k: (i, j)),  # C tile
        compiler_params=compiler_params,
        interpret=USE_INTERPRET,
    )(x, y)


# Test
a = random.normal(random.PRNGKey(0), (512, 512), dtype=jnp.float32)
b = random.normal(random.PRNGKey(1), (512, 512), dtype=jnp.float32)

result = pallas_matmul_k_tiled(a, b)
expected = a @ b
max_err = jnp.max(jnp.abs(result - expected))
print(f"K-tiled matmul: max error {max_err:.2e}")
print(f"Correct: {jnp.allclose(result, expected, atol=1e-3)}")


In [ ]:
# Benchmark: Pallas matmul vs XLA
print(f"Matmul benchmark on {backend.upper()}")
print("=" * 90)

jit_matmul = jax.jit(lambda a, b: a @ b)

for size in [512, 1024, 2048]:
    print(f"\nSize: {size} x {size}")
    print("-" * 90)
    a = random.normal(random.PRNGKey(0), (size, size), dtype=jnp.float32)
    b = random.normal(random.PRNGKey(1), (size, size), dtype=jnp.float32)

    benchmark(jit_matmul, a, b, label="jnp.matmul (XLA)")

    bk = min(128, size)
    bm = bn = min(128, size)
    try:
        benchmark(
            lambda a, b: pallas_matmul_k_tiled(a, b, bm=bm, bn=bn, bk=bk),
            a, b, label=f"Pallas K-tiled (bm=bn=bk={bk})"
        )
    except Exception as e:
        print(f"  Pallas: FAILED -- {type(e).__name__}: {e}")

print("\nNote: XLA's matmul is *extremely* optimized. Matching it is a sign")
print("that your kernel fundamentals are solid. Beating it is a PhD thesis.")

## Part 8: Advanced Kernel Techniques

Everything so far has been building to this. Now we'll cover the techniques
that separate toy kernels from production ones:

1. **Online softmax** -- the algorithmic key to Flash Attention
2. **Simplified Flash Attention** -- combine tiled matmul + online softmax into a single kernel
3. **Scratch space & `program_id`** -- TPU-specific tools for managing state across grid iterations
4. **Custom VJP** -- making kernels differentiable so they work in training

> **Production note:** For production attention, prefer `jax.nn.dot_product_attention` which supports cuDNN Flash Attention on GPU and optimized implementations on TPU. Writing your own is for learning and for custom patterns (e.g., custom masking, sparse attention) that the built-in doesn't support.

### Online Softmax: The Key to Flash Attention

Our fused softmax loads the **entire row** into SRAM, then computes max, exp, sum, divide.
But what if the row is 128K elements wide? It won't fit in SRAM.

**Online softmax** processes the row in chunks, maintaining running statistics.
The key insight is a rescaling trick:

```
Say you've processed chunks 0..j and computed:
   m_old = max of all elements seen so far
   l_old = sum of exp(x_i - m_old) for all i seen so far

Now you see chunk j+1 with new max m_chunk:
   m_new = max(m_old, m_chunk)

   # THE TRICK: rescale the old sum to the new max
   l_new = l_old * exp(m_old - m_new) + sum(exp(chunk - m_new))

   # exp(m_old - m_new) is always in [0, 1] since m_new >= m_old
   # This "corrects" all the old exponentials without re-computing them!
```

This algorithm (Milakov & Gimelshein, 2018) is the foundation of Flash Attention.
The same streaming reduction pattern also applies beyond attention -- for example,
Cut Cross-Entropy (Apple, ICLR 2025) uses it to compute cross-entropy loss over
large vocabularies without materializing the full logit matrix.

In [ ]:
# Online softmax: processes the row in chunks
#
# NOTE: This kernel loads the full row into VMEM for simplicity.
# The ALGORITHM is what matters here -- it processes chunks as if the
# full row didn't fit. The real payoff comes when this is embedded in
# Flash Attention, where only one K/V chunk is in SRAM at a time
# (via BlockSpec tiling -- see the TPU version in cell below).
#
# We reshape the row into (n_chunks, CHUNK) to avoid dynamic_slice,
# which is not supported in Pallas TPU lowering.

def online_softmax_kernel(x_ref, o_ref):
    x = x_ref[...]  # (block_rows, D)
    block_rows, D = x.shape
    CHUNK = 128
    n_chunks = D // CHUNK

    # Reshape to (block_rows, n_chunks, CHUNK) so we can iterate over chunks
    x_chunked = x.reshape(block_rows, n_chunks, CHUNK)

    # Running statistics
    m_init = jnp.full((block_rows, 1), -jnp.inf, dtype=jnp.float32)
    l_init = jnp.zeros((block_rows, 1), dtype=jnp.float32)

    def update_stats(i, carry):
        m_prev, l_prev = carry
        chunk = x_chunked[:, i, :]  # (block_rows, CHUNK)

        # Update running max
        m_chunk = jnp.max(chunk, axis=-1, keepdims=True)
        m_new = jnp.maximum(m_prev, m_chunk)

        # THE RESCALING TRICK: correct old sum for the new max
        alpha = jnp.exp(m_prev - m_new)
        l_new = l_prev * alpha + jnp.sum(jnp.exp(chunk - m_new), axis=-1, keepdims=True)

        return m_new, l_new

    m_final, l_final = jax.lax.fori_loop(0, n_chunks, update_stats, (m_init, l_init))

    # Final pass: compute output using the converged statistics
    o_ref[...] = jnp.exp(x - m_final) / l_final


def online_softmax(x, block_rows=8):
    n_rows, n_cols = x.shape
    assert n_cols % 128 == 0, "Columns must be divisible by chunk size (128)"
    return pl.pallas_call(
        online_softmax_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=True,  # fori_loop with dynamic indexing unsupported natively
    )(x)


# Verify: matches our fused softmax and jax.nn.softmax
key = random.PRNGKey(99)
x = random.normal(key, (512, 1024), dtype=jnp.float32)
result_online = online_softmax(x)
result_jax = jax.nn.softmax(x, axis=-1)

print(f"Online softmax -- max error vs JAX: {jnp.max(jnp.abs(result_online - result_jax)):.2e}")
print(f"Correct: {jnp.allclose(result_online, result_jax, atol=1e-5)}")

### Simplified Flash Attention

Now we combine everything into the kernel that revolutionized transformer inference
and training. Flash Attention fuses **three operations** into one kernel:

```
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d)) @ V
```

**Naive attention** materializes the full N x N score matrix in HBM:
- Memory: O(N^2) -- explodes for long sequences
- HBM traffic: read Q,K -> write scores -> read scores -> softmax -> write weights -> read weights,V -> write output

**Flash Attention** never materializes the N x N matrix:
- For each query block Q_i, iterate over K/V blocks
- Compute scores for one block at a time (fits in SRAM)
- Use **online softmax** to accumulate attention-weighted values
- Memory: O(N) -- scales to long sequences
- HBM traffic: one pass over Q, K, V -> one write of output

```
For each query block Q_i:
    m, l, o = -inf, 0, 0            # running max, sum, output

    For each K/V block (K_j, V_j):
        S = Q_i @ K_j^T / sqrt(d)   # scores: small (bq x bkv), stays in SRAM
        m_new = max(m, rowmax(S))
        alpha = exp(m - m_new)       # correction for old values
        P = exp(S - m_new)           # attention weights (unnormalized)
        l = l * alpha + rowsum(P)
        o = o * alpha + P @ V_j      # accumulate weighted values

    O_i = o / l                      # final normalization
```

> **What follows is a simplified version** that demonstrates the online softmax algorithm but loads all K/V into SRAM upfront (via `BlockSpec((N, d), ...)`). This means SRAM usage is O(N x d), **not** the O(bkv x d) of real Flash Attention. The TPU grid-based version later in this section tiles K/V properly. We start with this version because the algorithm is easier to read without scratch space and `pl.when` guards.

> **History:** [FlashAttention](https://arxiv.org/abs/2205.05538) (Dao et al., 2022) introduced this approach. [FlashAttention-2](https://arxiv.org/abs/2307.08691) (2023) improved parallelism and work partitioning. [FlashAttention-3](https://arxiv.org/abs/2407.08691) (2024) exploits Hopper GPU asynchrony (warp specialization, pingpong scheduling) to reach 75% H100 utilization, and adds FP8 support.

In [ ]:
# Simplified Flash Attention -- forward pass, no masking, single head
#
# IMPORTANT: This loads ALL of K,V into SRAM per query block.
# It demonstrates the online softmax ALGORITHM correctly, but does NOT
# achieve the O(1) SRAM usage of real Flash Attention.
# See the TPU grid-based version below for proper K/V tiling.

def flash_attn_kernel(q_ref, k_ref, v_ref, o_ref):
    """Online-softmax attention (simplified -- all K,V in SRAM).

    q_ref: (bq, d)  -- one query block (tiled via grid)
    k_ref: (N, d)   -- all keys (loaded fully -- NOT how real Flash Attention works)
    v_ref: (N, d)   -- all values
    o_ref: (bq, d)  -- output block
    """
    q = q_ref[...]          # (bq, d)
    k_all = k_ref[...]      # (N, d)
    v_all = v_ref[...]      # (N, d)

    bq, d = q.shape
    N = k_all.shape[0]
    bkv = 128               # process 128 keys at a time
    n_kv_blocks = N // bkv
    scale = jnp.float32(d) ** -0.5

    # Reshape K,V into blocks to avoid dynamic_slice (not supported on TPU)
    k_blocks = k_all.reshape(n_kv_blocks, bkv, d)  # (n_kv_blocks, bkv, d)
    v_blocks = v_all.reshape(n_kv_blocks, bkv, d)

    # Accumulators -- use -inf for correct identity element
    m = jnp.full((bq, 1), -jnp.inf, dtype=jnp.float32)
    l = jnp.zeros((bq, 1), dtype=jnp.float32)
    o = jnp.zeros((bq, d), dtype=jnp.float32)

    def body(j, carry):
        m_prev, l_prev, o_prev = carry

        k_j = k_blocks[j]   # (bkv, d)
        v_j = v_blocks[j]   # (bkv, d)

        # Attention scores (stays in SRAM -- never written to HBM)
        s = (q @ k_j.T) * scale              # (bq, bkv)

        # Online softmax update
        s_max = jnp.max(s, axis=-1, keepdims=True)
        m_new = jnp.maximum(m_prev, s_max)
        alpha = jnp.exp(m_prev - m_new)       # correction factor
        p = jnp.exp(s - m_new)                # unnormalized weights
        l_new = l_prev * alpha + jnp.sum(p, axis=-1, keepdims=True)

        # Accumulate weighted values (rescale old output for new max)
        o_new = o_prev * alpha + p @ v_j      # (bq, d)

        return m_new, l_new, o_new

    m, l, o = jax.lax.fori_loop(0, n_kv_blocks, body, (m, l, o))

    # Final normalization
    o_ref[...] = (o / l).astype(o_ref.dtype)


def flash_attention_simplified(q, k, v, bq=128):
    """Simplified attention via Pallas. Tiles over query blocks only."""
    N, d = q.shape
    assert N % bq == 0

    return pl.pallas_call(
        flash_attn_kernel,
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        grid=(N // bq,),
        in_specs=[
            pl.BlockSpec((bq, d), lambda i: (i, 0)),   # Q: tiled
            pl.BlockSpec((N, d), lambda i: (0, 0)),     # K: full (NOT ideal)
            pl.BlockSpec((N, d), lambda i: (0, 0)),     # V: full (NOT ideal)
        ],
        out_specs=pl.BlockSpec((bq, d), lambda i: (i, 0)),
        interpret=True,  # fori_loop with dynamic indexing unsupported natively
    )(q, k, v)

In [ ]:
# Verify correctness against naive attention
@jax.jit
def naive_attention(q, k, v):
    """Standard attention -- materializes the full NxN score matrix."""
    scores = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    weights = jax.nn.softmax(scores, axis=-1)
    return weights @ v


key = random.PRNGKey(0)
N, d = 512, 128  # sequence length, head dimension
q = random.normal(key, (N, d), dtype=jnp.float32)
k = random.normal(random.PRNGKey(1), (N, d), dtype=jnp.float32)
v = random.normal(random.PRNGKey(2), (N, d), dtype=jnp.float32)

result_flash = flash_attention_simplified(q, k, v)
result_naive = naive_attention(q, k, v)

max_err = jnp.max(jnp.abs(result_flash - result_naive))
print(f"Simplified flash attention vs naive:")
print(f"  Max error: {max_err:.2e}")
print(f"  Correct:   {jnp.allclose(result_flash, result_naive, atol=1e-3)}")

# Note: to add batch/multi-head dimensions to any Pallas kernel,
# use jax.vmap. For example:
#   batched_attn = jax.vmap(flash_attention_simplified)
# This maps the kernel over a leading batch dimension without
# changing the kernel code itself.

In [ ]:
# Benchmark: Simplified Flash Attention vs Naive
print(f"Attention benchmark on {backend.upper()}")
print("=" * 90)

for N in [256, 512, 1024, 2048]:
    d = 128
    print(f"\nN={N}, d={d}  (score matrix would be {N}x{N} = {N*N:,} elements)")
    print("-" * 90)

    q = random.normal(random.PRNGKey(0), (N, d), dtype=jnp.float32)
    k = random.normal(random.PRNGKey(1), (N, d), dtype=jnp.float32)
    v = random.normal(random.PRNGKey(2), (N, d), dtype=jnp.float32)

    benchmark(naive_attention, q, k, v, label="Naive (materializes NxN)")

    bq = min(128, N)
    try:
        benchmark(flash_attention_simplified, q, k, v, label=f"Simplified flash (bq={bq})")
    except Exception as e:
        print(f"  Simplified flash: FAILED -- {type(e).__name__}: {e}")

print()
print("NOTE: The 'simplified' kernel loads ALL of K,V into SRAM per query block,")
print("so it does NOT have the memory advantage of real Flash Attention.")
print("It may be slower than naive for small N due to fori_loop overhead.")
print("The TPU grid-based version below tiles K,V properly for O(1) SRAM usage.")

### TPU-Specific: Scratch Space & `program_id`

Two powerful Pallas APIs that unlock more sophisticated kernel patterns:

**`scratch_shapes`** -- allocate persistent buffers in VMEM that survive across grid iterations.
Unlike local variables (which reset each iteration), scratch space keeps state. Use it for
accumulators, running statistics, or double-buffering.

**`pl.program_id(axis)`** -- get the current grid index inside the kernel. Combined with
**`pl.when(condition)`**, this lets you run code conditionally (e.g., initialize on first
iteration, normalize on last).

Here's our Flash Attention rewritten to use the **grid** for K/V iteration instead of
`fori_loop`, with scratch space for the online softmax accumulators. This version lets
the TPU compiler pipeline K/V blocks (overlapping DMA with compute).


In [ ]:
if IS_TPU:
    def flash_attn_kernel_tpu(q_ref, k_ref, v_ref, o_ref,
                               m_scratch, l_scratch):
        """Flash Attention with grid-based K/V iteration.

        Grid dim 0: parallel (query blocks, independent)
        Grid dim 1: arbitrary (K/V blocks, reduction, pipelined by compiler)

        Scratch space holds running max (m) and sum (l) across K/V iterations.
        This is the REAL Flash Attention pattern: only one K/V block is in
        SRAM at a time, giving O(bkv * d) SRAM usage regardless of N.
        """
        q = q_ref[...]      # (bq, d)
        k = k_ref[...]      # (bkv, d), one K block (tiled by grid dim 1)
        v = v_ref[...]      # (bkv, d)

        bq, d = q.shape
        scale = jnp.float32(d) ** -0.5

        # Initialize accumulators on the first K/V block
        @pl.when(pl.program_id(1) == 0)
        def _init():
            m_scratch[...] = jnp.full(m_scratch.shape, -jnp.inf, dtype=jnp.float32)
            l_scratch[...] = jnp.zeros(l_scratch.shape, dtype=jnp.float32)
            o_ref[...] = jnp.zeros(o_ref.shape, dtype=o_ref.dtype)

        m_prev = m_scratch[...]
        l_prev = l_scratch[...]

        # Scores
        s = (q @ k.T) * scale

        # Online softmax
        s_max = jnp.max(s, axis=-1, keepdims=True)
        m_new = jnp.maximum(m_prev, s_max)
        alpha = jnp.exp(m_prev - m_new)
        p = jnp.exp(s - m_new)
        l_new = l_prev * alpha + jnp.sum(p, axis=-1, keepdims=True)

        # Accumulate (NOTE: if o_ref.dtype is bf16, this accumulates in bf16
        # which loses precision. Production kernels use fp32 scratch for o.)
        o_ref[...] = (o_ref[...] * alpha + p @ v).astype(o_ref.dtype)
        m_scratch[...] = m_new
        l_scratch[...] = l_new

        # Normalize on the last K/V block
        @pl.when(pl.program_id(1) == pl.num_programs(1) - 1)
        def _normalize():
            o_ref[...] = (o_ref[...] / l_scratch[...]).astype(o_ref.dtype)


    def flash_attention_tpu(q, k, v, bq=128, bkv=128):
        N, d = q.shape
        return pl.pallas_call(
            flash_attn_kernel_tpu,
            out_shape=jax.ShapeDtypeStruct((N, d), q.dtype),
            grid=(N // bq, N // bkv),
            in_specs=[
                pl.BlockSpec((bq, d), lambda i, j: (i, 0)),     # Q: tiled by i
                pl.BlockSpec((bkv, d), lambda i, j: (j, 0)),    # K: tiled by j
                pl.BlockSpec((bkv, d), lambda i, j: (j, 0)),    # V: tiled by j
            ],
            out_specs=pl.BlockSpec((bq, d), lambda i, j: (i, 0)),
            scratch_shapes=[
                pltpu.VMEM((bq, 1), jnp.float32),   # m_scratch (running max)
                pltpu.VMEM((bq, 1), jnp.float32),   # l_scratch (running sum)
            ],
            compiler_params=pltpu.CompilerParams(
                dimension_semantics=["parallel", "arbitrary"],
            ),
        )(q, k, v)


    # Verify
    N, d = 512, 128
    q = random.normal(random.PRNGKey(0), (N, d), dtype=jnp.float32)
    k = random.normal(random.PRNGKey(1), (N, d), dtype=jnp.float32)
    v = random.normal(random.PRNGKey(2), (N, d), dtype=jnp.float32)

    result_tpu = flash_attention_tpu(q, k, v)
    result_ref = naive_attention(q, k, v)
    max_err = jnp.max(jnp.abs(result_tpu - result_ref))
    print(f"Flash Attention (TPU grid version, real K/V tiling)")
    print(f"  Max error: {max_err:.2e}")
    print(f"  Correct:   {jnp.allclose(result_tpu, result_ref, atol=1e-3)}")

    # Benchmark both versions
    print(f"\nComparing simplified vs grid-based Flash Attention:")
    print("-" * 90)
    N = 1024
    q = random.normal(random.PRNGKey(0), (N, d), dtype=jnp.float32)
    k = random.normal(random.PRNGKey(1), (N, d), dtype=jnp.float32)
    v = random.normal(random.PRNGKey(2), (N, d), dtype=jnp.float32)

    benchmark(flash_attention_simplified, q, k, v, label="Simplified (fori_loop, all K/V in SRAM)")
    benchmark(flash_attention_tpu, q, k, v, label="Grid + scratch (pipelined, real tiling)")
    benchmark(naive_attention, q, k, v, label="Naive (baseline)")
else:
    print("(TPU-specific cell: demonstrates scratch_shapes, program_id, pl.when)")
    print("(Switch to TPU runtime to run this)")
    print()
    print("Key APIs shown here:")
    print("  scratch_shapes=[pltpu.VMEM(shape, dtype)]    persistent SRAM buffers")
    print("  pl.program_id(axis)                          current grid index")
    print("  @pl.when(condition)                          conditional execution")
    print("  pltpu.CompilerParams(dimension_semantics=...) pipeline control")

### Making Kernels Trainable: `custom_vjp`

JAX cannot automatically differentiate through Pallas kernels -- they're opaque
compiled code. To use a kernel during training (where you need gradients), wrap
it with `jax.custom_vjp` and provide the backward pass yourself.

The pattern is always the same:
1. **Forward:** run the kernel, save values needed for the backward pass ("residuals")
2. **Backward:** given the output gradient, compute the input gradient using the residuals

For softmax, the gradient has a clean closed form:
```
d_softmax/d_x = softmax(x) * (d_out - sum(d_out * softmax(x), axis=-1))
```


In [ ]:
# Wrap our fused softmax with a custom backward pass

@jax.custom_vjp
def trainable_softmax(x):
    """Pallas softmax that supports jax.grad."""
    return fused_softmax(x)


def trainable_softmax_fwd(x):
    out = fused_softmax(x)
    return out, (out,)    # (output, residuals for backward)


def trainable_softmax_bwd(residuals, g):
    out, = residuals
    # Softmax gradient: out * (g - sum(g * out, axis=-1, keepdims=True))
    s = jnp.sum(g * out, axis=-1, keepdims=True)
    return (out * (g - s),)


trainable_softmax.defvjp(trainable_softmax_fwd, trainable_softmax_bwd)


# Verify: gradient matches JAX's built-in softmax gradient
key = random.PRNGKey(0)
x = random.normal(key, (64, 128), dtype=jnp.float32)

grad_ours = jax.grad(lambda x: jnp.sum(trainable_softmax(x)))(x)
grad_jax = jax.grad(lambda x: jnp.sum(jax.nn.softmax(x, axis=-1)))(x)

print("custom_vjp: making Pallas kernels differentiable")
print(f"  Gradient shape: {grad_ours.shape}")
print(f"  Max grad error: {jnp.max(jnp.abs(grad_ours - grad_jax)):.2e}")
print(f"  Correct:        {jnp.allclose(grad_ours, grad_jax, atol=1e-4)}")
print()
print("This pattern works for any Pallas kernel -- you just need to")
print("provide the math for the backward pass.")


### Exercise -- Fused LayerNorm + Residual + Dropout

In transformers, LayerNorm is almost always followed by a **residual connection** and sometimes **dropout**: `output = dropout(LayerNorm(x)) + x`. This is a fusion opportunity -- instead of writing intermediate results to HBM, do everything in one kernel.

Start with just the residual: `output = LayerNorm(x) * weight + bias + x`. For an extra challenge, add dropout using a PRNG key (see `jax.random.bernoulli`).

In [ ]:
# ============================================================
# YOUR CODE HERE: Fuse LayerNorm + residual connection
# output = LayerNorm(x) * weight + bias + x
# ============================================================

def layernorm_residual_kernel(x_ref, weight_ref, bias_ref, o_ref):
    x = x_ref[...]
    w = weight_ref[...]
    b = bias_ref[...]
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.mean(jnp.square(x - mean), axis=-1, keepdims=True)
    x_norm = (x - mean) / jnp.sqrt(var + 1e-5)
    o_ref[...] = x_norm * w + b + x

def fused_layernorm_residual(x, weight, bias, block_rows=8):
    n_rows, n_cols = x.shape
    return pl.pallas_call(
        layernorm_residual_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[
            pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
            pl.BlockSpec((n_cols,), lambda i: (0,)),
            pl.BlockSpec((n_cols,), lambda i: (0,)),
        ],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=USE_INTERPRET,
    )(x, weight, bias)


# ---------- test harness (requires jax_layernorm from Part 6, cell 30) ----------
key = random.PRNGKey(42)
x = random.normal(key, (1024, 512), dtype=jnp.float32)
w = random.normal(random.PRNGKey(1), (512,), dtype=jnp.float32)
b = random.normal(random.PRNGKey(2), (512,), dtype=jnp.float32)

try:
    result = fused_layernorm_residual(x, w, b)
    expected = jax_layernorm(x, w, b) + x  # LN + residual
    if jnp.allclose(result, expected, atol=1e-4):
        print("Correct! Fused LayerNorm + residual works.")
    else:
        max_err = jnp.max(jnp.abs(result - expected))
        print(f"Wrong answer. Max error: {max_err:.2e}")
except NotImplementedError:
    print("Implement layernorm_residual_kernel and fused_layernorm_residual above.")

#### Solution

```python
def layernorm_residual_kernel(x_ref, weight_ref, bias_ref, o_ref):
    x = x_ref[...]
    w = weight_ref[...]
    b = bias_ref[...]
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.mean(jnp.square(x - mean), axis=-1, keepdims=True)
    x_norm = (x - mean) / jnp.sqrt(var + 1e-5)
    o_ref[...] = x_norm * w + b + x   # fused residual add

def fused_layernorm_residual(x, weight, bias, block_rows=8):
    n_rows, n_cols = x.shape
    return pl.pallas_call(
        layernorm_residual_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[
            pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
            pl.BlockSpec((n_cols,), lambda i: (0,)),
            pl.BlockSpec((n_cols,), lambda i: (0,)),
        ],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=USE_INTERPRET,
    )(x, weight, bias)
```

### Common Gotchas

Before you move on, here are the mistakes that trip up most people learning Pallas:

| Gotcha | Details |
|--------|---------|
| **Pallas kernels are not auto-differentiable** | JAX cannot `jax.grad` through `pallas_call`. You must write a `custom_vjp` (see Part 8). Forgetting this and putting a kernel in a training loop gives a cryptic error. |
| **`interpret=True` hides real bugs** | Kernels that work in interpret mode may fail when compiled for TPU/GPU: unsupported ops, dtype restrictions, memory layout issues. Always test on real hardware. |
| **`=` vs `+=` in reduction kernels** | `o_ref[...] = x @ y` overwrites the output each iteration. For reductions (K-tiled matmul), you need `o_ref[...] += x @ y`. This works because Pallas zero-initializes the output buffer. |
| **Input sizes must divide evenly** | All kernels in this notebook `assert n % block_size == 0`. In production, you need masking (`jnp.where`) or padding for non-divisible sizes. Pallas does not handle this automatically. |
| **All shapes must be static** | Pallas requires shapes known at trace time. Variable-length sequences need padding to a fixed max length. |
| **VMEM/shared memory capacity** | If your block size is too large, the kernel fails to compile (TPU) or runs out of shared memory (GPU) with obscure errors. Rule of thumb: total tile data should stay under ~50% of available SRAM. |
| **`jax.vmap` for batch dimensions** | Don't change your kernel to handle batches -- just `vmap` it: `batched_fn = jax.vmap(pallas_fn)`. |
| **Debugging: use `jax.debug.print`** | Inside a kernel with `interpret=True`, `jax.debug.print("{x}", x=tile)` prints intermediate values. Essential for debugging wrong outputs. |

## Part 9: Exercises to Go Further

You now know the Pallas programming model and have written real kernels. Here are challenges in increasing difficulty. Each exercise has a code cell with stubs -- fill in the TODOs and run the test harness to check your work.

### Exercise 9.1: Fused softmax + scale (easy)
Modify the softmax kernel to also multiply by `1/sqrt(d_k)` -- the scaling factor in attention. No extra memory traffic since it fuses into the existing kernel.

In [ ]:
# ============================================================
# EXERCISE 9.1: Fused softmax + scale
# YOUR CODE HERE
# ============================================================

def scaled_softmax_kernel(x_ref, o_ref, *, scale):
    x = x_ref[...] * scale
    row_max = jnp.max(x, axis=-1, keepdims=True)
    safe = x - row_max
    numerator = jnp.exp(safe)
    denominator = jnp.sum(numerator, axis=-1, keepdims=True)
    o_ref[...] = numerator / denominator

def pallas_scaled_softmax(x, d_k, block_rows=8):
    n_rows, n_cols = x.shape
    # Use a plain Python float -- Pallas kernels cannot capture JAX arrays
    # as constants (they're opaque to the compiler). Python floats are fine
    # because they become literal constants in the compiled kernel.
    import math
    scale = 1.0 / float(jnp.sqrt(d_k))
    return pl.pallas_call(
        functools.partial(scaled_softmax_kernel, scale=scale),
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(n_rows // block_rows,),
        in_specs=[pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((block_rows, n_cols), lambda i: (i, 0)),
        interpret=USE_INTERPRET,
    )(x)


# ---------- test harness (don't modify) ----------
key = random.PRNGKey(42)
x = random.normal(key, (1024, 512), dtype=jnp.float32)
d_k = 64  # typical head dimension

try:
    result = pallas_scaled_softmax(x, d_k)
    expected = jax.nn.softmax(x / jnp.sqrt(d_k))
    if jnp.allclose(result, expected, atol=1e-5):
        print("Correct! Fused scaled softmax works.")
    else:
        max_err = jnp.max(jnp.abs(result - expected))
        print(f"Wrong answer. Max error: {max_err:.2e}")
except NotImplementedError:
    print("Implement scaled_softmax_kernel and pallas_scaled_softmax above.")


### Exercise 9.2: Block-size autotuning (medium)
Write a function that tries multiple block-size configurations for a given kernel and input shape, then returns the fastest one. Think of it as a mini `triton.autotune`.

In [ ]:
# ============================================================
# EXERCISE 9.2: Block-size autotuning
# YOUR CODE HERE
# ============================================================

def autotune(kernel_fn, x, block_sizes=(8, 16, 32, 64, 128), warmup=5, trials=20):
    best_bs = None
    best_time = float("inf")
    for bs in block_sizes:
        if x.shape[0] % bs != 0:
            continue
        for _ in range(warmup):
            kernel_fn(x, block_rows=bs).block_until_ready()
        times = []
        for _ in range(trials):
            start = time.perf_counter()
            kernel_fn(x, block_rows=bs).block_until_ready()
            times.append((time.perf_counter() - start) * 1000)
        avg = np.median(times)
        if avg < best_time:
            best_time = avg
            best_bs = bs
    return best_bs, best_time


# ---------- test harness (don't modify) ----------
key = random.PRNGKey(42)
x = random.normal(key, (4096, 1024), dtype=jnp.float32)

try:
    best_bs, best_time = autotune(fused_softmax, x)
    print(f"Best block size: {best_bs}, time: {best_time:.3f} ms")
except NotImplementedError:
    print("Implement autotune above.")

### Exercise 9.3: Fused bias + GeLU + matmul (hard)
Fuse bias addition and GeLU activation into the matmul kernel. This is the core pattern in transformer MLP layers: `GeLU(X @ W + b)`.

The key insight: instead of writing `X @ W` to HBM, adding bias, writing again, applying GeLU, and writing a third time -- do it all in the kernel that computes the matmul tile.

In [ ]:
# ============================================================
# EXERCISE 9.3: Fused bias + GeLU + matmul
# YOUR CODE HERE
# ============================================================

def gelu(x):
    return 0.5 * x * (1 + jnp.tanh(jnp.sqrt(2 / jnp.pi) * (x + 0.044715 * x ** 3)))

def matmul_bias_gelu_kernel(x_ref, w_ref, bias_ref, o_ref):
    tile = x_ref[...] @ w_ref[...]
    tile = tile + bias_ref[...]
    o_ref[...] = gelu(tile)

def pallas_matmul_bias_gelu(x, w, bias, bm=128, bn=128):
    m, k = x.shape
    _, n = w.shape
    return pl.pallas_call(
        matmul_bias_gelu_kernel,
        out_shape=jax.ShapeDtypeStruct((m, n), x.dtype),
        grid=(m // bm, n // bn),
        in_specs=[
            pl.BlockSpec((bm, k), lambda i, j: (i, 0)),
            pl.BlockSpec((k, bn), lambda i, j: (0, j)),
            pl.BlockSpec((bn,), lambda i, j: (j,)),
        ],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j: (i, j)),
        interpret=True,  # 1D bias BlockSpec has TPU layout issues
    )(x, w, bias)


# ---------- test harness (don't modify) ----------
key = random.PRNGKey(0)
x = random.normal(key, (512, 256), dtype=jnp.float32)
w = random.normal(random.PRNGKey(1), (256, 512), dtype=jnp.float32)
bias = random.normal(random.PRNGKey(2), (512,), dtype=jnp.float32)

try:
    result = pallas_matmul_bias_gelu(x, w, bias)
    expected = gelu(x @ w + bias)
    max_err = jnp.max(jnp.abs(result - expected))
    if jnp.allclose(result, expected, atol=1e-3):
        print(f"Correct! Fused matmul+bias+GeLU works. Max error: {max_err:.2e}")
    else:
        print(f"Wrong answer. Max error: {max_err:.2e}")
except NotImplementedError:
    print("Implement matmul_bias_gelu_kernel and pallas_matmul_bias_gelu above.")

## Supplementary materials

I put together a [NotebookLM notebook](https://notebooklm.google.com/notebook/c37e48ea-9fba-486c-95ec-dee82f35d3c8) loaded with this codelab, all the referenced papers, and the Pallas docs. You can use it to ask questions as you work through the exercises.

I also generated some video overviews that walk through the key concepts:

| # | Topic | Covers |
|---|-------|--------|
| 1 | [Why Kernels Matter](https://notebooklm.google.com/notebook/c37e48ea-9fba-486c-95ec-dee82f35d3c8?artifactId=620a0aab-b6ae-4bf9-9956-ece8c4263ec0) | Parts 0-3: the mental model, tiling, BlockSpecs |
| 2 | [Kernel Fusion and Benchmarking](https://notebooklm.google.com/notebook/c37e48ea-9fba-486c-95ec-dee82f35d3c8?artifactId=1ef49655-d413-4d86-847e-8e55a2e4d6f9) | Parts 4-5: fused softmax, memory bandwidth, benchmarking |
| 3 | [Hardware-Aware Optimization](https://notebooklm.google.com/notebook/c37e48ea-9fba-486c-95ec-dee82f35d3c8?artifactId=3b1a4dca-b5a1-4798-bcc5-139dd04ba89d) | Part 6: roofline model, MXU, pipelining, LayerNorm |
| 4 | [Matrix Multiplication](https://notebooklm.google.com/notebook/c37e48ea-9fba-486c-95ec-dee82f35d3c8?artifactId=57977328-b6f0-42c2-a3a3-e609c4cede72) | Part 7: tiled matmul, K-tiling, the accumulator pattern |
| 5 | [Flash Attention from Scratch](https://notebooklm.google.com/notebook/c37e48ea-9fba-486c-95ec-dee82f35d3c8?artifactId=0ded5b60-b1ed-4f77-ac0b-23b1b56a9118) | Part 8: online softmax, Flash Attention, custom_vjp |

## Part 10: Resources

- [JAX Pallas documentation](https://docs.jax.dev/en/latest/pallas/index.html) -- official docs, API reference, tutorials
- [Pallas changelog](https://docs.jax.dev/en/latest/pallas/CHANGELOG.html) -- track API changes across JAX versions
- [Pallas TPU examples](https://docs.jax.dev/en/latest/pallas/tpu/examples.html) -- matmul, flash attention on TPU
- [Triton tutorials](https://triton-lang.org/main/getting-started/tutorials/index.html) -- useful GPU kernel intuition (Pallas supports Triton as an alternative GPU backend)
- [`jax.nn.dot_product_attention`](https://docs.jax.dev/en/latest/_autosummary/jax.nn.dot_product_attention.html) -- production flash attention in JAX (cuDNN on GPU, optimized on TPU)
- [FlashAttention paper](https://arxiv.org/abs/2205.05538) -- the paper that made kernel fusion mainstream in ML
- [FlashAttention-3](https://arxiv.org/abs/2407.08691) -- exploits asynchrony on Hopper GPUs
- [GPU MODE lectures](https://www.youtube.com/@GPUMODE) -- community lectures on GPU programming
- [What Every Programmer Should Know About Memory](https://people.freebsd.org/~lstewart/articles/cpumemory.pdf) -- the memory hierarchy concepts that make kernel optimization click